# LSTM Inference Walkthrough

Training and inference are different jobs. Training learns model parameters. Inference loads already-learned parameters and produces a forecast without changing the model.

This notebook walks through the saved LSTM inference path.

## What inference does not use

During inference there are no target labels, no loss, no gradients, no backpropagation, no optimizer steps, no epochs, and no early stopping. The model is put into `eval()` mode and predictions run under `torch.no_grad()`.

In [1]:
# ruff: noqa: E402, I001
import json
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from market_resonance.inference import count_parameters, run_lstm_inference
from market_resonance.inference.lstm_inference import load_model_from_checkpoint

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "treasury_yields_daily.csv"
CHECKPOINT_PATH = PROJECT_ROOT / "reports" / "models" / "first_lstm.pt"
METRICS_PATH = PROJECT_ROOT / "results" / "inference_metrics.json"

## Load the saved checkpoint

The checkpoint contains learned model weights plus the metadata needed for inference, including feature columns, lookback length, horizon, and training normalization statistics.

In [2]:
model, metadata = load_model_from_checkpoint(CHECKPOINT_PATH)
{
    "parameter_count": count_parameters(model),
    "lookback": metadata["lookback"],
    "horizon": metadata["horizon"],
    "num_features": len(metadata["feature_columns"]),
    "num_targets": len(metadata["target_columns"]),
    "model_training_mode": model.training,
}

{'parameter_count': 24519,
 'lookback': 60,
 'horizon': 1,
 'num_features': 28,
 'num_targets': 7,
 'model_training_mode': False}

`model.training` should be `False`, which means `model.eval()` has been applied.

## Run inference

The inference command rebuilds the latest 60-day feature sequence, applies the saved training normalization statistics, runs the model with no gradients, and saves forecast diagnostics.

In [3]:
result = run_lstm_inference(
    data_path=DATA_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    metrics_path=METRICS_PATH,
)
{
    "latest_input_date": result.latest_input_date,
    "forecast_date": result.forecast_date,
    "parameter_count": result.parameter_count,
    "single_sample_latency_ms": result.single_sample_latency_ms,
    "batch_latency_ms": result.batch_latency_ms,
}

{'latest_input_date': '2026-09-08',
 'forecast_date': '2026-09-08 + 1 trading day(s)',
 'parameter_count': 24519,
 'single_sample_latency_ms': 0.5181,
 'batch_latency_ms': 7.5259}

## Seven-maturity forecast

The model predicts future yield changes in percentage points. The inference layer converts those changes into basis points and adds them to the latest observed yields.

In [4]:
forecast = pd.DataFrame(result.forecast)
forecast

,maturity,latest_yield_percent,predicted_change_bp,forecast_yield_percent
0,3M,3.94,0.2308,3.9423
1,6M,4.00,0.5275,4.0053
2,1Y,4.15,-0.6766,4.1432
3,2Y,4.39,1.1872,4.4019
4,5Y,4.57,-0.1153,4.5688
5,10Y,4.80,0.9126,4.8091
6,30Y,5.25,1.1722,5.2617


## Metrics JSON

The same forecast and runtime diagnostics are saved for reproducibility.

In [5]:
metrics = json.loads(METRICS_PATH.read_text())
metrics

{'forecast_date': '2026-09-08 + 1 trading day(s)',
 'latest_input_date': '2026-09-08',
 'forecast': [{'maturity': '3M',
   'latest_yield_percent': 3.94,
   'predicted_change_bp': 0.2308,
   'forecast_yield_percent': 3.9423},
  {'maturity': '6M',
   'latest_yield_percent': 4.0,
   'predicted_change_bp': 0.5275,
   'forecast_yield_percent': 4.0053},
  {'maturity': '1Y',
   'latest_yield_percent': 4.15,
   'predicted_change_bp': -0.6766,
   'forecast_yield_percent': 4.1432},
  {'maturity': '2Y',
   'latest_yield_percent': 4.39,
   'predicted_change_bp': 1.1872,
   'forecast_yield_percent': 4.4019},
  {'maturity': '5Y',
   'latest_yield_percent': 4.57,
   'predicted_change_bp': -0.1153,
   'forecast_yield_percent': 4.5688},
  {'maturity': '10Y',
   'latest_yield_percent': 4.8,
   'predicted_change_bp': 0.9126,
   'forecast_yield_percent': 4.8091},
  {'maturity': '30Y',
   'latest_yield_percent': 5.25,
   'predicted_change_bp': 1.1722,
   'forecast_yield_percent': 5.2617}],
 'parameter_coun

## Why `torch.no_grad()` matters

In training, PyTorch tracks operations so it can compute gradients. In inference, we do not need gradients because no parameters are updated. `torch.no_grad()` makes prediction faster and uses less memory.

In [6]:
with torch.no_grad():
    no_grad_is_enabled = not torch.is_grad_enabled()

no_grad_is_enabled

True